In [1]:
from tcr_toolbox.sequencing_analysis.align_count import count_umi_cell
import glob
import matplotlib as mpl
from tcr_toolbox.utils.plot_utils import startfig, set_mpl_params, plot_count_histogram
mpl, mylines = set_mpl_params(mpl=mpl)
from tcr_toolbox.sequencing_analysis_PRIVATE.plate_tcr_epi_count_analysis import init_umi_count_analysis_dir, read_umi_count_tsv, filter_UMI_counts_df_dict, plot_detected_tcr_epi_plate_map, write_tcr_epi_pair_well_count_files, read_gdna_counts_csv, read_tcr_epi_pair_counts_csv, calculate_tcr_epi_pair_exp_prob_matrix, obs_tcr_epi_count_binom_sf_test, add_reactive_epi_and_tcr_co_occurrence_bool_col, add_tcr_epi_pairs_only_supported_by_co_occurrence_bool_col, plot_tcr_epi_pair_ma_plot, plot_pvalue_corrected_rank_vs_pvalue_corrected, write_reactive_pair_results, eval_performance_run, plot_and_combine_screen_pvals_stouffer
from tcr_toolbox.sequencing_analysis.utils import overlap_between_ref_and_count_names
from tcr_toolbox.utils.stat_utils import normalize_counts_df_by_total_counts
import numpy as np
from pathlib import Path
import shutil

In [ ]:
project_dir = Path('/data/T100_vs_Ag100_mel-pt-a')

In [ ]:
init_umi_count_analysis_dir(project_dir)

## Preprocessing

In [3]:
counts_df_dict = read_umi_count_tsv(project_dir = project_dir,
                                    plate_name_split_location = 2
                                    )

Reading plate: 4-2-3
Reading plate: 4-2-1
Reading plate: 4-1-1
Reading plate: 4-1-4
Reading plate: 4-1-2
Reading plate: 4-1-3
Reading plate: 4-2-2


In [4]:
epi_umi_threshold_dict = {
    "4-2-3":50,
    "4-2-1":50,
    "4-1-1":60,
    "4-1-4":70, 
    "4-1-2":100,
    "4-1-3":100,
    "4-2-2":100
                          }

tcr_umi_threshold_dict = {
    "4-2-3":100,
    "4-2-1":100,
    "4-1-1":40,
    "4-1-4":50, 
    "4-1-2":30,
    "4-1-3":60,
    "4-2-2":70
}

In [ ]:
counts_df_dict = filter_UMI_counts_df_dict(
    project_dir = project_dir, 
    counts_df_dict = counts_df_dict, 
    epi_umi_threshold=epi_umi_threshold_dict, 
    tcr_umi_threshold=tcr_umi_threshold_dict,
    min_start_fraction_of_top_umi_count=0.9,
    min_end_fraction_of_top_umi_count = 0.1,
    num_bins=50
)

Filtering plate: 4-2-3
Number of unique epis: 84
Number of unique tcrs: 39
Filtering plate: 4-2-1
Number of unique epis: 126
Number of unique tcrs: 75
Filtering plate: 4-1-1
Number of unique epis: 158
Number of unique tcrs: 71
Filtering plate: 4-1-4
Number of unique epis: 106
Number of unique tcrs: 52
Filtering plate: 4-1-2
Number of unique epis: 138
Number of unique tcrs: 56
Filtering plate: 4-1-3
Number of unique epis: 116
Number of unique tcrs: 46
Filtering plate: 4-2-2
Number of unique epis: 88
Number of unique tcrs: 61


In [ ]:
# After individual TCR-Ag pair validation co-cultures, code can be re-run with a validating_pair_dict to color annotate dots in results plots. 
validating_pair_dict = {'epi_mel-pt-a_CCSER2_1384-tcr_9_17': True,
                               'epi_mel-pt-a_GFPT2_896-tcr_27_41': True,
                               'epi_mel-pt-a_CCSER2_1384-tcr_44_80': True,
                               'epi_mel-pt-a_CCSER2_1384-tcr_76_150': True,
                               'epi_mel-pt-a_CCSER2_1384-tcr_96_188': True,

                               'epi_mel-pt-a_CCSER2_1384-tcr_5_7': True,
                               'epi_mel-pt-a_CCSER2_1384-tcr_40_76': True,
                               'epi_mel-pt-a_CCSER2_1384-tcr_8_11': True,

                               'epi_mel-pt-a_GFPT2_896-tcr_18_27': True,
                               'epi_mel-pt-a_ADAM10_1753-tcr_15_24': True, 

                               'epi_mel-pt-a_XRCC6_2333-tcr_95_186': False, 
                                'epi_mel-pt-a_FUT8_1672A-tcr_69_132': False, 
}

In [7]:
plot_detected_tcr_epi_plate_map(project_dir = project_dir,
                                counts_df_dict = counts_df_dict,
                                validating_pair_dict = validating_pair_dict,
                                epitope_barcode_length = 18,
                                detected_fontsize = 3)

Plotting plate: 4-2-3
Plotting plate: 4-2-1
Plotting plate: 4-1-1
Plotting plate: 4-1-4
Plotting plate: 4-1-2
Plotting plate: 4-1-3
Plotting plate: 4-2-2


In [10]:
list(counts_df_dict.keys())

['4-2-3', '4-2-1', '4-1-1', '4-1-4', '4-1-2', '4-1-3', '4-2-2']

In [ ]:
plates_list = ['4-1-1', '4-1-2', '4-1-3', '4-1-4']

print(plates_list)

write_tcr_epi_pair_well_count_files(project_dir = project_dir,
                                    plates_list = plates_list,
                                    plates_list_name = 'mel-pt-a',
                                    counts_df_dict = counts_df_dict,
                                    collapse_epitope_barcode = True,
                                    epitope_barcode_length= 18,
                                    validating_pair_dict = validating_pair_dict
                                   )

## Bulk seq data QC

In [ ]:
bulk_project_dir = Path('/data/T100_vs_Ag100_Run-4_01_02_2024_bulk/Ag')
name_split_location = 2

In [ ]:
for count_file in (bulk_project_dir / "counts").glob("*.csv"):
    print(count_file.name)

    overlap_between_ref_and_count_names(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        reference_file='/references/T200_Ag200/Ag100.fa',
        library_ref_id_list=None,
        print_diff_names=True,
    )

    outs_dir = Path(bulk_project_dir) / "outs"
    outs_dir.mkdir(parents=True, exist_ok=True)

    plot_count_histogram(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        count_file_count_col="read_count",
        bin_min=0,
        bin_max=5,
        bin_size=0.1,
        log10 = True, 
        xlabel="Number of reads per transcript",
        percentile_ratio_filter_threshold = 2, 
        title=count_file.stem.split("_")[name_split_location],
        save_file_path=outs_dir / f"{count_file.stem.split('_')[name_split_location]}_count_hist.pdf",
        ylim_max=50,
        w=6.25,
        h=6.25,
    )
    print("\n")

7627_9_Ag100-2x-puro-new-bulk_GCTTGTCA-GTATGTTC_S9_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv
# intersecting: 200
# diff: 2
# union: 202
Sensitivity: 99.01% (200/202 reference names found in counts)
refs not in count names: {'p20_stuffer', 'GLC'}
count names not in ref subset: set()
95th percentile: 8200.35 | 5th percentile: 993.05 | 95th/5th ratio: 8.26


7627_12_Ag100-2x-puro-old-bulk_AAGTCCAA-TACTCATA_S12_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv
# intersecting: 200
# diff: 2
# union: 202
Sensitivity: 99.01% (200/202 reference names found in counts)
refs not in count names: {'p20_stuffer', 'GLC'}
count names not in ref subset: set()
95th percentile: 6331.35 | 5th percentile: 812.90 | 95th/5th ratio: 7.79


7627_11_Ag100-1x-puro-sorted-bulk_GGACTTGG-CGTCTGCG_S11_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv
# intersecting: 200
# diff: 2
# union: 202
Sensitivity: 99.01% (200/202 reference names found in counts)
refs not in count n

In [ ]:
bulk_project_dir = Path('/data/bulk_seq_data/TCR')
name_split_location = 2

In [ ]:
for count_file in (bulk_project_dir / "counts").glob("*.csv"):
    print(count_file.name)

    overlap_between_ref_and_count_names(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        reference_file='/references/T200_Ag200/T100_beta.fa',
        library_ref_id_list=None,
        print_diff_names=True,
    )

    outs_dir = Path(bulk_project_dir) / "outs"
    outs_dir.mkdir(parents=True, exist_ok=True)

    plot_count_histogram(
        count_file=count_file,
        count_file_ref_name_col="reference_name",
        count_file_count_col="read_count",
        bin_min=0,
        bin_max=5,
        bin_size=0.1,
        log10 = True, 
        xlabel="Number of reads per transcript",
        percentile_ratio_filter_threshold = 2, 
        title=count_file.stem.split("_")[name_split_location],
        save_file_path=outs_dir / f"{count_file.stem.split('_')[name_split_location]}_count_hist.pdf",
        ylim_max=50,
        w=6.25,
        h=6.25,
    )
    print("\n")

7627_13_T100-bulk_TTACAGGA-GCTTGTCA_S13_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv
# intersecting: 92
# diff: 1
# union: 93
Sensitivity: 98.92% (92/93 reference names found in counts)
refs not in count names: {'102_198'}
count names not in ref subset: set()
95th percentile: 16399.25 | 5th percentile: 3444.35 | 95th/5th ratio: 4.76


7627_14_T100-sorted-bulk_GGCATTCT-CAAGCTAG_S14_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv
# intersecting: 92
# diff: 1
# union: 93
Sensitivity: 98.92% (92/93 reference names found in counts)
refs not in count names: {'102_198'}
count names not in ref subset: set()
95th percentile: 11259.70 | 5th percentile: 2104.40 | 95th/5th ratio: 5.35




## PAIR-Scan analysis

In [ ]:
epi_bulk_counts_df = read_gdna_counts_csv(count_file = '/data/bulk_seq_data/Ag/counts/7627_12_Ag100-2x-puro-old-bulk_AAGTCCAA-TACTCATA_S12_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv',
                                          count_file_ref_name_col = 'reference_name',
                                          collapse_epitope_barcode = True,
                                          epitope_barcode_length = 18
                                          )

In [10]:
normalized_epi_bulk_counts_df = normalize_counts_df_by_total_counts(count_df = epi_bulk_counts_df, count_col = 'read_count')

In [ ]:
tcr_bulk_counts_df = read_gdna_counts_csv(count_file = '/data/bulk_seq_data/TCR/counts/7627_13_T100-bulk_TTACAGGA-GCTTGTCA_S13_R1_001_trimmed_sorted_2ndfiltered_cigarmd_filtered_counts.csv',
                                         count_file_ref_name_col = 'reference_name',
                                         )

In [12]:
normalized_tcr_bulk_counts_df = normalize_counts_df_by_total_counts(count_df = tcr_bulk_counts_df, count_col = 'read_count')

In [ ]:
pair_count_matrix_df = read_tcr_epi_pair_counts_csv(pair_count_matrix_csv = project_dir / "outs" / "mel-pt-a" /"pair_count_matrix_df_mel-pt-a.csv")

In [ ]:
exp_pair_prob_matrix_df, pair_count_matrix_df = calculate_tcr_epi_pair_exp_prob_matrix(pair_count_matrix_df = pair_count_matrix_df,
                                                                                       normalized_epitope_bulk_counts_df = normalized_epi_bulk_counts_df,
                                                                                       normalized_tcr_bulk_counts_df = normalized_tcr_bulk_counts_df,
                                                                                       bulk_epitope_count_col = 'read_count',
                                                                                       bulk_tcr_count_col = 'read_count',
                                                                                       normalize_using_plate = False
                                                                                       )

In [ ]:
alpha = 0.0005

In [ ]:
pair_fold_changes_df = obs_tcr_epi_count_binom_sf_test(project_dir = project_dir,
                                                       pair_count_matrix_df = pair_count_matrix_df,
                                                       exp_pair_prob_matrix_df = exp_pair_prob_matrix_df,
                                                       plates_list_name = 'mel-pt-a',
                                                       validating_pair_dict = validating_pair_dict,
                                                       fdr_correction = True,
                                                       alpha = alpha,
                                                       normalize_total_called_pairs = False,
                                                       normalize_total_wells_with_detected_pair = True,
                                                       sort_fold_change = True,
                                                       sort_pvalue = False
                                                      )

710 TCR-epitope pairs have non-zero well counts of 7968 tested TCR-epitope pairs


In [ ]:
pair_fold_changes_df = add_tcr_epi_pairs_only_supported_by_co_occurrence_bool_col(project_dir = project_dir,
                                                                                  plates_list_name = 'mel-pt-a',
                                                                                  pair_fold_changes_df = pair_fold_changes_df,
                                                                                  epi_transcript_ratio = 0.7,
                                                                                  tcr_transcript_ratio = 0.7
                                                                                 )

In [18]:
sensitivity, sensitivity_without_co_occurrence_filter, precision = eval_performance_run(pair_fold_changes_df = pair_fold_changes_df,
                                                                                        validating_pair_dict = validating_pair_dict)
print(sensitivity, sensitivity_without_co_occurrence_filter, precision)

1.0 1.0 1.0


In [ ]:
pair_fold_changes_df.to_excel(project_dir / "outs" / "mel-pt-a" / "pair_fold_changes_df_mel-pt-a.xlsx")

In [21]:
pair_fold_changes_df['Fold_change'].max()

762.5388788920993

In [ ]:
plot_tcr_epi_pair_ma_plot(pair_fold_changes_df = pair_fold_changes_df,
                          title = 'mel-pt-a',
                          xlim_max = 0.0006,
                          xticks_list = list(np.arange(-0.0001, 0.0006, 0.0002)),
                          xlim_min = -0.0001,
                          ylim_max = 900,
                          save_dir = project_dir / "outs" / "mel-pt-a", 
                          custom_pair_color_dict = None,
                          plot_filter_only_supported_co_occurrence = True,
                          plot_reactive_pair_co_occurrence = False,
                          annotate = True,
                          annotation_fontsize = 2.5,
                          w = 6,
                          h = 6
                         )

In [ ]:
plot_tcr_epi_pair_ma_plot(pair_fold_changes_df = pair_fold_changes_df,
                          title = 'mel-pt-a',
                          xlim_max = 0.0006,
                          xticks_list = list(np.arange(-0.0001, 0.0006, 0.0002)),
                          xlim_min = -0.0001,
                          ylim_max = 900,
                          save_dir = project_dir / "outs" / "mel-pt-a",
                          custom_pair_color_dict = None,
                          plot_filter_only_supported_co_occurrence = True,
                          plot_reactive_pair_co_occurrence = False,
                          annotate = False,
                          #annotation_fontsize = 2,
                          w = 6,
                          h = 6
                         )

In [ ]:
plot_pvalue_corrected_rank_vs_pvalue_corrected(
    pair_fold_changes_df=pair_fold_changes_df,
    save_dir=Path(project_dir) / "outs" / "mel-pt-a",
    title="mel-pt-a",
    xlim_max=70,
    xlim_min=-5,
    ylim_max=155,
    ylim_min=-10,
    adj_pval_alpha = None,
    filter_significant_with_more_than_one_epi=True,
    custom_pair_color_dict=None,
    annotate=True,
)

In [ ]:
plot_pvalue_corrected_rank_vs_pvalue_corrected(
    pair_fold_changes_df=pair_fold_changes_df,
    save_dir=Path(project_dir) / "outs" / "mel-pt-a",
    title="mel-pt-a",
    xlim_max=70,
    xlim_min=-5,
    ylim_max=155,
    ylim_min=-10,
    adj_pval_alpha = None,
    filter_significant_with_more_than_one_epi=True,
    custom_pair_color_dict=None,
    annotate=False,
)

In [ ]:
write_reactive_pair_results(pair_fold_changes_df = pair_fold_changes_df,
                            project_dir = project_dir,
                            plates_list_name = 'mel-pt-a',
                            validating_pair_dict = validating_pair_dict,
                           )